# XGBoost Model

XGBoost (Extreme Gradient Boosting) is an ensemble learning algorithm that builds
decision trees sequentially, where each new tree corrects the errors of the previous trees.

In this notebook:
- We train the XGBoost model
- Apply threshold tuning
- Evaluate model performance

In [1]:
import joblib
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, confusion_matrix, roc_auc_score
)

In [19]:
X_train = joblib.load("../models/X_train.pkl")
X_test  = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test  = joblib.load("../models/y_test.pkl")
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Class dist:", np.bincount(y_train))

Train: (565424, 51) | Test: (141357, 51)
Class dist: [557378   8046]


In [20]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=6,      # midpoint between 3 and 12
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.5,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
joblib.dump(xgb, "../models/xgboost_model.pkl")
print("XGBoost saved.")

XGBoost saved.


We use a custom threshold (0.6) instead of the default 0.5
to balance Precision and Recall.

In [21]:
y_proba = xgb.predict_proba(X_test)[:, 1]

print(f"{'Thresh':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 45)
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.30, 0.91, 0.05):
    yp = (y_proba >= t).astype(int)
    pr = precision_score(y_test, yp, zero_division=0)
    re = recall_score(y_test, yp, zero_division=0)
    f1 = f1_score(y_test, yp, zero_division=0)
    print(f"{t:>8.2f} {accuracy_score(y_test,yp):>8.3f} {pr:>8.3f} {re:>8.3f} {f1:>8.3f}")
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"\nPeak F1: {best_f1:.3f} at threshold {best_thresh:.2f}")
print(f"RF F1=0.541 — XGB peak above RF: {best_f1 > 0.541}")

  Thresh      Acc     Prec      Rec       F1
---------------------------------------------
    0.30    0.987    0.558    0.537    0.547
    0.35    0.988    0.619    0.476    0.538
    0.40    0.989    0.679    0.427    0.524
    0.45    0.989    0.726    0.374    0.494
    0.50    0.989    0.754    0.315    0.444
    0.55    0.988    0.779    0.258    0.388
    0.60    0.988    0.812    0.210    0.334
    0.65    0.988    0.820    0.163    0.272
    0.70    0.987    0.850    0.124    0.217
    0.75    0.987    0.867    0.085    0.154
    0.80    0.986    0.867    0.055    0.104
    0.85    0.986    0.894    0.029    0.057
    0.90    0.986    0.920    0.011    0.023

Peak F1: 0.547 at threshold 0.30
RF F1=0.541 — XGB peak above RF: True


In [22]:
THRESHOLD = best_thresh
y_pred    = (y_proba >= THRESHOLD).astype(int)

print(f"===== XGBoost Results (threshold={THRESHOLD:.2f}) =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

joblib.dump(THRESHOLD, "../models/xgb_threshold.pkl")
print(f"\nRF F1=0.541 — XGB above RF: {round(f1_score(y_test,y_pred,zero_division=0),3) > 0.541}")

===== XGBoost Results (threshold=0.30) =====
Accuracy : 0.987
Precision: 0.558
Recall   : 0.537
F1 Score : 0.547
ROC-AUC  : 0.951
Confusion Matrix:
[[138490    856]
 [   931   1080]]

RF F1=0.541 — XGB above RF: True
